# Baseline — Classifying (Corrupted) Pet Images

**Competition:** classify each pet image into one of 12 anonymous classes
(source: Oxford-IIIT Pet). The twist: **test images may be corrupted / shifted in
distribution** compared to training images, so robust preprocessing matters.

- **Task:** 12-class image classification (420 train / 240 test images, 224×224 RGB)
- **Metric:** accuracy
- **Kaggle link:** _TODO: add link_

**Approach:** with only 420 training images, training a CNN from scratch would badly
overfit. Instead we use **transfer learning**: a ResNet-18 pretrained on ImageNet as a
frozen feature extractor, plus a simple Logistic Regression on top. This runs in
minutes on CPU.

In [1]:
import numpy as np
import pandas as pd
import torch
import torchvision.transforms as T
from torchvision.models import resnet18, ResNet18_Weights
from PIL import Image

DATA_DIR = "."   # folder containing train.csv / train_images/ ...

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
print(train.shape, test.shape)
train.head(3)

(420, 3) (240, 2)


,id,image_path,label
0,train_00000,train_images/train_00000.jpg,class_07
1,train_00001,train_images/train_00001.jpg,class_03
2,train_00002,train_images/train_00002.jpg,class_09


In [2]:
# Frozen pretrained backbone -> 512-d features per image
device = "cuda" if torch.cuda.is_available() else "cpu"
weights = ResNet18_Weights.IMAGENET1K_V1
backbone = resnet18(weights=weights)
backbone.fc = torch.nn.Identity()          # drop the classification head
backbone.eval().to(device)

preprocess = weights.transforms()          # resize + normalize exactly as in pretraining

@torch.no_grad()
def extract_features(paths, batch_size=32):
    feats = []
    for i in range(0, len(paths), batch_size):
        batch = [preprocess(Image.open(f"{DATA_DIR}/{p}").convert("RGB"))
                 for p in paths[i:i+batch_size]]
        feats.append(backbone(torch.stack(batch).to(device)).cpu().numpy())
    return np.vstack(feats)

X_train = extract_features(train["image_path"].tolist())
X_test  = extract_features(test["image_path"].tolist())
print(X_train.shape, X_test.shape)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


  0%|          | 0.00/44.7M [00:00<?, ?B/s]

 13%|█▎        | 5.75M/44.7M [00:00<00:00, 60.0MB/s]

 36%|███▌      | 15.9M/44.7M [00:00<00:00, 86.8MB/s]

 54%|█████▍    | 24.2M/44.7M [00:00<00:00, 84.9MB/s]

 79%|███████▊  | 35.1M/44.7M [00:00<00:00, 96.1MB/s]

100%|██████████| 44.7M/44.7M [00:00<00:00, 97.9MB/s]

(420, 512) (240, 512)


In [3]:
# Simple classifier on top of the frozen features
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

y = train["label"].values
clf = LogisticRegression(max_iter=2000, C=1.0)

scores = cross_val_score(clf, X_train, y, cv=5, scoring="accuracy")
print(f"5-fold CV accuracy: {scores.mean():.4f} +/- {scores.std():.4f}")

5-fold CV accuracy: 0.8619 +/- 0.0297


In [4]:
clf.fit(X_train, y)
sub = pd.DataFrame({"id": test["id"], "label": clf.predict(X_test)})
sub.to_csv("submission.csv", index=False)
sub.head()

,id,label
0,test_00000,class_05
1,test_00001,class_04
2,test_00002,class_01
3,test_00003,class_07
4,test_00004,class_05


## Ideas to improve

- **Fine-tune** the whole ResNet (unfreeze) with augmentation on a GPU (Kaggle/Colab).
- Use a stronger backbone: ResNet-50, EfficientNet, or a **CLIP/DINO-v2** encoder.
- **Test-time augmentation** (average predictions over flips/crops) — helps a lot when
  test images are corrupted.
- Add training-time corruption augmentations (blur, noise, JPEG artifacts) so the
  model sees at train time what it will face at test time.
